# Lab 02 · Parte 3 — Principio multiplicativo: outfits de una tienda online (R)

Una tienda de moda online quiere generar recomendaciones automáticas de **outfits completos**: una camiseta, un pantalón y un par de zapatos.

| Categoría | Prendas | Opciones |
|---|---|---|
| Camisetas | C1, C2, C3 | 3 |
| Pantalones | P1, P2 | 2 |
| Zapatos | Z1, Z2 | 2 |

Un outfit es un elemento del **producto cartesiano** $C \times P \times Z$: una terna ordenada (camiseta, pantalón, zapatos).

In [1]:
prendas <- list(
  camiseta = c("C1", "C2", "C3"),
  pantalon = c("P1", "P2"),
  zapatos  = c("Z1", "Z2")
)

## Parte 1. Principio multiplicativo

### 1–2. Cálculo paso a paso

El **principio multiplicativo** dice que si una elección se hace en varias etapas independientes, el número total de resultados es el **producto** de las opciones de cada etapa.

1. **Elegir la camiseta:** hay **3** opciones.
2. **Elegir el pantalón:** para *cada* camiseta hay **2** pantalones, así que hay $3 \times 2 = 6$ combinaciones camiseta + pantalón.
3. **Elegir los zapatos:** para *cada* una de esas 6 combinaciones hay **2** pares de zapatos, así que hay $6 \times 2 = 12$ outfits.

$$|C \times P \times Z| = |C| \cdot |P| \cdot |Z| = 3 \cdot 2 \cdot 2 = 12$$

El diagrama de árbol de la celda siguiente muestra las tres etapas: cada nivel multiplica las ramas del anterior.

In [2]:
opciones <- lengths(prendas)
total_teorico <- prod(opciones)

acumulado <- cumprod(opciones)
for (paso in seq_along(opciones)) {
  cat(sprintf("Paso %d: elegir %-8s -> %d opciones -> %d combinaciones\n",
              paso, names(opciones)[paso], opciones[paso], acumulado[paso]))
}
cat(sprintf("\nTotal: %s = %d outfits\n", paste(opciones, collapse = " x "), total_teorico))

Paso 1: elegir camiseta -> 3 opciones -> 3 combinaciones
Paso 2: elegir pantalon -> 2 opciones -> 6 combinaciones
Paso 3: elegir zapatos  -> 2 opciones -> 12 combinaciones

Total: 3 x 2 x 2 = 12 outfits


In [3]:
# Diagrama de árbol: cada nivel es una etapa de la elección
for (c in prendas$camiseta) {
  cat(c, "\n")
  for (i in seq_along(prendas$pantalon)) {
    p <- prendas$pantalon[i]
    ultimo_p <- i == length(prendas$pantalon)
    cat(sprintf("  %s %s\n", if (ultimo_p) "└──" else "├──", p))
    for (j in seq_along(prendas$zapatos)) {
      z <- prendas$zapatos[j]
      rama_z <- if (j == length(prendas$zapatos)) "└──" else "├──"
      sangria <- if (ultimo_p) "    " else "│   "
      cat(sprintf("  %s%s %s  ->  (%s, %s, %s)\n", sangria, rama_z, z, c, p, z))
    }
  }
}

C1 
  ├── P1
  │   ├── Z1  ->  (C1, P1, Z1)
  │   └── Z2  ->  (C1, P1, Z2)
  └── P2
      ├── Z1  ->  (C1, P2, Z1)
      └── Z2  ->  (C1, P2, Z2)
C2 
  ├── P1
  │   ├── Z1  ->  (C2, P1, Z1)
  │   └── Z2  ->  (C2, P1, Z2)
  └── P2
      ├── Z1  ->  (C2, P2, Z1)
      └── Z2  ->  (C2, P2, Z2)
C3 
  ├── P1
  │   ├── Z1  ->  (C3, P1, Z1)
  │   └── Z2  ->  (C3, P1, Z2)
  └── P2
      ├── Z1  ->  (C3, P2, Z1)
      └── Z2  ->  (C3, P2, Z2)


### 3. Interpretación en un sistema de recomendación

- El catálogo de **7 prendas** genera **12 outfits distintos**: el sistema tiene 12 recomendaciones posibles, casi el doble de opciones que prendas.
- El número de outfits **crece multiplicando, no sumando**: si se añade una cuarta camiseta, los outfits pasan de 12 a $4 \cdot 2 \cdot 2 = 16$, sin tocar el resto del catálogo. Con un catálogo real (por ejemplo, 200 camisetas, 80 pantalones y 50 zapatos) serían 800 000 combinaciones.
- Por eso un recomendador real **no puede enseñarlas todas**: necesita filtrar y ordenar (por estilo, colores, talla, stock o historial del cliente). Saber cuántas combinaciones hay permite estimar el coste de evaluarlas todas antes de diseñar el sistema.

## Parte 2. Generación de combinaciones

### 1. Generar todos los outfits

`expand.grid()` calcula el producto cartesiano. Varía primero el primer argumento, así que se pasan las categorías en orden inverso y luego se reordenan las columnas para que la camiseta sea la que cambia más despacio.

In [4]:
outfits <- expand.grid(zapatos = prendas$zapatos,
                       pantalon = prendas$pantalon,
                       camiseta = prendas$camiseta,
                       stringsAsFactors = FALSE)[, c("camiseta", "pantalon", "zapatos")]

lista_outfits <- apply(outfits, 1, function(o) sprintf("(%s, %s, %s)", o[1], o[2], o[3]))
cat(sprintf("%2d. %s", seq_along(lista_outfits), lista_outfits), sep = "\n")

 1. (C1, P1, Z1)
 2. (C1, P1, Z2)
 3. (C1, P2, Z1)
 4. (C1, P2, Z2)
 5. (C2, P1, Z1)
 6. (C2, P1, Z2)
 7. (C2, P2, Z1)
 8. (C2, P2, Z2)
 9. (C3, P1, Z1)
10. (C3, P1, Z2)
11. (C3, P2, Z1)
12. (C3, P2, Z2)


### 2. Representación en tabla

In [5]:
rownames(outfits) <- paste("Outfit", seq_len(nrow(outfits)))
outfits

,camiseta,pantalon,zapatos
,<chr>,<chr>,<chr>
Outfit 1,C1,P1,Z1
Outfit 2,C1,P1,Z2
Outfit 3,C1,P2,Z1
Outfit 4,C1,P2,Z2
Outfit 5,C2,P1,Z1
Outfit 6,C2,P1,Z2
Outfit 7,C2,P2,Z1
Outfit 8,C2,P2,Z2
Outfit 9,C3,P1,Z1


### 3. Verificación

El número de combinaciones generadas debe coincidir con el principio multiplicativo, y ninguna debe repetirse.

In [6]:
cat("Combinaciones generadas :", nrow(outfits), "\n")
cat("Combinaciones distintas :", nrow(unique(outfits)), "\n")
cat("Principio multiplicativo:", total_teorico, "\n")

stopifnot(nrow(outfits) == total_teorico, nrow(unique(outfits)) == total_teorico)
cat("\nVerificación correcta: coinciden.\n")

Combinaciones generadas : 12 
Combinaciones distintas : 12 
Principio multiplicativo: 12 

Verificación correcta: coinciden.


In [7]:
# Cada prenda aparece en tantos outfits como combinaciones haya del resto de categorías
lapply(outfits, table)

$camiseta

C1 C2 C3 
 4  4  4 

$pantalon

P1 P2 
 6  6 

$zapatos

Z1 Z2 
 6  6 


## Conclusiones

- Con 3 camisetas, 2 pantalones y 2 pares de zapatos se forman **12 outfits** ($3 \cdot 2 \cdot 2$).
- La generación exhaustiva produce exactamente **12 combinaciones distintas**, lo que verifica el principio multiplicativo.
- Cada prenda aparece en tantos outfits como combinaciones haya del resto de categorías: cada camiseta en $2 \cdot 2 = 4$ outfits y cada pantalón o par de zapatos en $3 \cdot 2 = 6$.